# Analyst-Ready SEC Filings RAG System

**Course:** AI Engineering  
**Professor:** Apostolos Filippas  
**Institution:** Fordham University  
**Team:** Joel Ebukatobi, Claudia Gisemba

This notebook builds a complete SEC 10-K Retrieval-Augmented Generation pipeline for analyst workflows.

In [14]:
%load_ext autoreload
%autoreload 2

In [64]:
# Section 1: Setup & Configuration
import os
import json
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
from dotenv import load_dotenv

from src.ingest import load_raw_filings, filing_stats, load_live_filings
from src.chunk import parse_sections, section_coverage_stats, build_chunks, chunk_stats
from src.embed import embed_chunks, embedding_info
from src.retrieve import HybridRetriever
from src.generate import compare, generate_structured_output
from src.evaluate import build_test_set, run_retrieval_eval, ablation_table

load_dotenv()
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
if not OPENAI_API_KEY:
    print("Warning: OPENAI_API_KEY is not set. Generation sections will fail until set.")
else:
    print("OpenAI API key loaded successfully.")

# Global constants for a reproducible project configuration.
CHUNK_SIZE = 800
CHUNK_OVERLAP = 200
TOP_K = 5
EMBED_MODEL = "all-MiniLM-L6-v2"
LLM_MODEL = "gpt-4o-mini"
# TICKERS = ["TSLA", "AAPL", "MSFT", "AMZN", "NVDA"]
# YEAR_RANGE = (2018, 2023)
# TARGET_SECTIONS = ["section_1a", "section_7", "section_3"]

Path("data").mkdir(parents=True, exist_ok=True)
print("Constants defined and environment loaded.")

OpenAI API key loaded successfully.
Constants defined and environment loaded.


In [50]:
TICKERS = ["TSLA", "AAPL", "AMZN", "NVDA", "CCL"]
YEAR_RANGE = (2018, 2023)
TARGET_SECTIONS = ["section_1a", "section_7", "section_3"]

## Section 2: Data Ingestion
Load 10-K filings from Hugging Face and normalize into a DataFrame with core metadata.

In [52]:
from src.ingest import load_live_filings, filing_stats

In [53]:
import inspect
print(inspect.getsource(load_live_filings))

def load_live_filings(tickers, year_range):
    rows = []
    year_min, year_max = year_range

    for ticker in tqdm(tickers, desc="Processing Tickers"):
        try:
            company = Company(ticker)
            filings = company.get_filings(form="10-K")
            if not filings: continue

            for filing in filings:
                f_year = filing.filing_date.year
                if year_min <= f_year <= year_max:
                    
                    # 1. Get the structured TenK object
                    tenk = filing.obj()
                    
                    # 2. Extract Sections (Item 1A, 7, and 8)
                    # We use .get() or dictionary access to avoid AttributeErrors
                    risk_factors = getattr(tenk, "risk_factors", "") or tenk["Item 1A"] or ""
                    mda = getattr(tenk, "management_discussion", "") or tenk["Item 7"] or ""
                    
                    # Item 8 (Financials) is where the Debt Tables live
    

In [ ]:
# NOTE: An earlier version used load_raw_filings() from the HuggingFace edgar-corpus dataset.
# That approach was replaced by load_live_filings() which pulls directly from SEC EDGAR
# via the edgartools library, giving us structured section access (Items 1A, 3, 7, 8).

In [55]:
df_filings = load_live_filings(
    tickers=TICKERS,
    year_range=YEAR_RANGE
)

print("Raw filing DataFrame shape:", df_filings.shape)
display(df_filings.head(3))
display(filing_stats(df_filings).head(20))

Processing Tickers:   0%|          | 0/2 [00:00<?, ?it/s]TenK falling back to legacy parser for 'Item 1A' (filing: 0001564590-22-016871). New parser sections available: ['part_iii_item_15', 'part_iii_item_10', 'part_iii_item_11', 'part_iii_item_12', 'part_iii_item_13', 'part_iii_item_14']. This fallback will be removed in v6.0.
TenK falling back to legacy parser for 'Item 1A' (filing: 0001564590-22-016871). New parser sections available: ['part_iii_item_15', 'part_iii_item_10', 'part_iii_item_11', 'part_iii_item_12', 'part_iii_item_13', 'part_iii_item_14']. This fallback will be removed in v6.0.
TenK falling back to legacy parser for 'Item 7' (filing: 0001564590-22-016871). New parser sections available: ['part_iii_item_15', 'part_iii_item_10', 'part_iii_item_11', 'part_iii_item_12', 'part_iii_item_13', 'part_iii_item_14']. This fallback will be removed in v6.0.
TenK falling back to legacy parser for 'Item 7' (filing: 0001564590-22-016871). New parser sections available: ['part_iii_ite

Raw filing DataFrame shape: (13, 7)


,ticker,year,company,section_1a,section_7,section_8,section_3
0,TSLA,2022,"Tesla, Inc.",,,,
1,TSLA,2022,"Tesla, Inc.",ITEM 1A.\tRISK FACTORS\n\n You should careful...,ITEM 7.\tMANAGEMENT’S DISCUSSION AND ANALYSIS ...,ITEM 8.\t\tFINANCIAL STATEMENTS AND SUPPLEMENT...,ITEM 3.\tLEGAL PROCEEDINGS\n\n For a descript...
2,TSLA,2021,"Tesla, Inc.",,,,


,ticker,year,count
0,NVDA,2018,1
1,NVDA,2019,1
2,NVDA,2020,1
3,NVDA,2021,1
4,NVDA,2022,1
5,TSLA,2018,1
6,TSLA,2019,1
7,TSLA,2020,2
8,TSLA,2021,2
9,TSLA,2022,2


## Section 3: Section Parsing
Extract Item 1A (Risk Factors), Item 7 (MD&A), and Item 3 (Legal Proceedings) from filing text.

In [65]:
df_sections = parse_sections(df_filings, target_sections=TARGET_SECTIONS)
print("Parsed sections shape:", df_sections.shape)
display(df_sections.head(5))
display(section_coverage_stats(df_sections))

Parsing sections: 100%|██████████| 13/13 [00:00<00:00, 22.43it/s]


Parsed sections shape: (30, 4)


,ticker,year,section_type,section_text
0,TSLA,2022,section_1a,ITEM 1A. RISK FACTORS You should carefully con...
1,TSLA,2022,section_7,ITEM 7. MANAGEMENT’S DISCUSSION AND ANALYSIS O...
2,TSLA,2022,section_3,ITEM 3. LEGAL PROCEEDINGS For a description of...
3,TSLA,2021,section_1a,You should carefully consider the risks descri...
4,TSLA,2021,section_7,The following discussion and analysis should b...


,section_type,coverage
0,section_1a,10
1,section_3,10
2,section_7,10


## Section 4: Section-Aware Chunking
Chunk each parsed section independently so chunks never cross section boundaries.

In [66]:
all_chunks = build_chunks(
    df_sections,
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    cache_path="data/chunks.pkl",
    use_multiprocessing=True
)

print("Total chunk count:", len(all_chunks))
display(pd.DataFrame(all_chunks).head(3))
display(chunk_stats(all_chunks))

Chunking sections: 100%|██████████| 30/30 [00:00<00:00, 82.48it/s]


Total chunk count: 368


,ticker,year,section_type,chunk_id,source,text
0,TSLA,2022,section_1a,TSLA-2022-section_1a-0,TSLA_2022_10k,ITEM 1A. RISK FACTORS You should carefully con...
1,TSLA,2022,section_1a,TSLA-2022-section_1a-1,TSLA_2022_10k,features for our products. There is no guarant...
2,TSLA,2022,section_1a,TSLA-2022-section_1a-2,TSLA_2022_10k,deliveries to customers’ homes and workplaces ...


,metric,value
0,count,368.00
1,min_tokens,32.00
2,mean_tokens,765.57
3,max_tokens,800.00


## Section 5: Embedding
Generate semantic vectors for every chunk and cache to disk.

In [68]:
vectors, vector_metadata = embed_chunks(
    all_chunks,
    model_name=EMBED_MODEL,
    batch_size=256,
    cache_path="data/embeddings.pkl"
)

print("Vectors shape:", getattr(vectors, "shape", None))
print("Embedding info:", embedding_info(vectors))

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2068.03it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 2/2 [00:58<00:00, 29.03s/it]

Vectors shape: (368, 384)
Embedding info: {'count': 368, 'dimension': 384, 'workers': 8}


## Section 6: Hybrid Retrieval
Combine FAISS semantic retrieval with BM25 lexical retrieval using weighted reranking.

In [70]:
retriever = HybridRetriever(
    vectors=vectors,
    metadata=vector_metadata,
    embed_model_name=EMBED_MODEL,
    semantic_weight=0.6,
    lexical_weight=0.4
)

sample_queries = [
    "major macroeconomic risk factors",
    "changes in segment revenue drivers",
    "notable legal proceedings and litigation"
]

for q in sample_queries:
    results = retriever.retrieve(query=q, ticker="TSLA", year=2022, section_type="section_1a", top_k=TOP_K)
    print("\nQuery:", q)
    display(pd.DataFrame(results)[["chunk_id", "semantic_score", "bm25_score", "final_score"]].head(TOP_K))

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4007.99it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Query: major macroeconomic risk factors


,chunk_id,semantic_score,bm25_score,final_score
0,TSLA-2022-section_1a-0,0.495252,1.000000,0.697151
1,TSLA-2022-section_1a-19,0.191694,0.447224,0.293906
2,TSLA-2022-section_1a-6,0.341360,0.154295,0.266534
3,TSLA-2022-section_1a-15,0.303501,0.191353,0.258642
4,TSLA-2022-section_1a-16,0.000000,0.601519,0.240607



Query: changes in segment revenue drivers


,chunk_id,semantic_score,bm25_score,final_score
0,TSLA-2022-section_1a-18,0.250871,1.000000,0.550522
1,TSLA-2022-section_1a-5,0.237557,0.658352,0.405875
2,TSLA-2022-section_1a-21,0.370442,0.000889,0.222621
3,TSLA-2022-section_1a-17,0.306594,0.085026,0.217967
4,TSLA-2022-section_1a-8,0.325019,0.052739,0.216107



Query: notable legal proceedings and litigation


,chunk_id,semantic_score,bm25_score,final_score
0,TSLA-2022-section_1a-19,0.162971,1.000000,0.497783
1,TSLA-2022-section_1a-16,0.233935,0.624004,0.389962
2,TSLA-2022-section_1a-18,0.185492,0.672231,0.380187
3,TSLA-2022-section_1a-15,0.147765,0.617939,0.335835
4,TSLA-2022-section_1a-20,0.183737,0.510054,0.314264


## Section 7: Comparative Reasoning
Retrieve evidence from two filing years and ask GPT-4o for structured differences.

In [72]:
comparison_result = compare(
    retriever=retriever,
    ticker="TSLA",
    section_type="section_1a",
    year_a=2020,
    year_b=2023,
    query="material risk changes over time",
    top_k=TOP_K,
    model=LLM_MODEL
)

print("Raw diff output:")
print(comparison_result.get("raw_diff", "")[:3000])

Raw diff output:
### Findings from SEC Filing Excerpts Comparison (2020 vs. 2023)

#### 1) New Items
- **Chunk ID TSLA-2023-section_1a-XX**: New references to specific regulatory changes affecting electric vehicle incentives and the impact of geopolitical tensions on supply chains were introduced, indicating a shift in focus towards external factors influencing operations.

#### 2) Removed Items
- **Chunk ID TSLA-2020-section_1a-3**: The mention of "perceptions about electric vehicle features" and "concerns about our future viability" has been omitted, suggesting a potential shift in confidence or market perception.
- **Chunk ID TSLA-2020-section_1a-4**: Specific references to product liability claims and self-insurance strategies have been removed, indicating a possible change in risk management strategy.

#### 3) Expanded Items
- **Chunk ID TSLA-2020-section_1a-7**: The discussion around software dependencies has been expanded to include more detailed examples of software issues and 

## Section 8: Structured Output Generation
Convert reasoning output into strict JSON schema and validate it with graceful error handling.

In [73]:
structured_output = generate_structured_output(comparison_result, model=LLM_MODEL)

if "error" in structured_output:
    print("Structured output failed gracefully:")
    print(json.dumps(structured_output, indent=2))
else:
    print("Validated structured JSON output:")
    print(json.dumps(structured_output, indent=2)[:5000])

Validated structured JSON output:
{
  "company": "Tesla",
  "filing_year_a": 2020,
  "filing_year_b": 2023,
  "risk_changes": [
    {
      "risk": "Regulatory changes affecting electric vehicle incentives",
      "change_type": "New",
      "evidence": "New references to specific regulatory changes affecting electric vehicle incentives and the impact of geopolitical tensions on supply chains were introduced.",
      "confidence": 0.8
    },
    {
      "risk": "Perceptions about electric vehicle features",
      "change_type": "Removed",
      "evidence": "The mention of 'perceptions about electric vehicle features' and 'concerns about our future viability' has been omitted.",
      "confidence": 0.7
    },
    {
      "risk": "Product liability claims and self-insurance strategies",
      "change_type": "Removed",
      "evidence": "Specific references to product liability claims and self-insurance strategies have been removed.",
      "confidence": 0.7
    },
    {
      "risk": "So

## Section 9: Evaluation
Evaluate retrieval and structured generation quality. Include ablation-style comparison table.

In [74]:
test_years = list(range(YEAR_RANGE[0], YEAR_RANGE[1] + 1))
test_set = build_test_set(tickers=TICKERS, years=test_years)
eval_results = run_retrieval_eval(retriever, test_set, top_k=TOP_K)

# Aggregate core retrieval metrics.
summary = pd.DataFrame({
    "Recall@5": [eval_results["Recall@5"].mean()],
    "MRR": [eval_results["MRR"].mean()]
})

print("Retrieval evaluation summary:")
display(summary)

# Real ablation: re-runs eval with BM25 and metadata filtering toggled off.
ablations = ablation_table(retriever, test_set, top_k=TOP_K)
display(ablations)

# Structured output and citation quality checks from one sample output.
from src.evaluate import citation_grounding_score, structured_output_accuracy

if isinstance(structured_output, dict) and "error" not in structured_output:
    cg = citation_grounding_score(structured_output)
    sa = structured_output_accuracy(structured_output)
else:
    cg, sa = 0.0, 0.0

print({
    "Citation Grounding Score": cg,
    "Structured Output Accuracy": sa
})

Running retrieval eval: 100%|██████████| 20/20 [00:00<00:00, 70.72it/s] 

Retrieval evaluation summary:


,Recall@5,MRR
0,0.4,0.4


,setup,Recall@5,MRR
0,Hybrid + Metadata + Rerank,0.40,0.40
1,No BM25,0.35,0.36
2,No Metadata Filtering,0.32,0.34
3,No Reranker,0.37,0.37


{'Citation Grounding Score': 1.0, 'Structured Output Accuracy': 1.0}


## Section 10: Streamlit App
The full pipeline is wired in app.py for deployment on Streamlit Community Cloud.

In [ ]:
# Run this command in terminal (not inside notebook):
# streamlit run app.py

print("Streamlit app entrypoint: app.py")
print("UI includes ticker, years, section, spinner, structured output, and raw JSON toggle.")